<font size=10>**TASK 3 - ??**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *Straining the great southern Melting Pot*

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

*«notebook description»*

**Question**: *??*

<font color = 'red'>**IR3 – Named Entity Recognition of Culinary and Spatial Themes**
Can we automatically extract dishes and locations from restaurant reviews using Named Entity Recognition (NER), and do the patterns of these entities reveal meaningful insights about cuisine types and dining experiences?
 
(Named Entity Recognition – focusing on dishes and locations, with optional analysis of entity transitions)

----

**IR3 – Dish Entity Networks and Cuisine Clusters**
 
Can we automatically extract dishes and related entities (e.g., locations) from restaurant reviews, and do the co-occurrence patterns of these entities form meaningful clusters that reveal cuisine types and cultural links? 

(NER + Co-occurrence Analysis + Clustering)</font>

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data](#2)
- [3. Named Entity Rcognition](#3)
    - [3.1 Specific Data Preparation](#3_1)
    - [3.2 Model Implementation](#3_2)
    - [3.3 Model Evaluation](#3_3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os
import pandas as pd
import spacy
from spacy import displacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import *
from visualizations import *
from general_preprocessing import *

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [3]:
dataset_original = load_dataset('../data/atlanta_restaurant_slice_2023.csv')

In [4]:
dataset_original.head()

,title,categoryName,website,url,reviewsCount,stars,text
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...


In [5]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53566 entries, 0 to 53565
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         53566 non-null  object 
 1   categoryName  53566 non-null  object 
 2   website       50600 non-null  object 
 3   url           53566 non-null  object 
 4   reviewsCount  53566 non-null  int64  
 5   stars         53566 non-null  float64
 6   text          53566 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 2.9+ MB


| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [6]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. Named Entity Recognition**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Specific Data Preparation</font> <a class="anchor" id="3_1"></a>
  
[Back to TOC](#toc)

In [7]:
# Tokenization and POS tagging using project pipeline
dataset["pos_list"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'
                                                            ))

dataset["tokens"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True
                                                            ))


In [8]:
dataset['features'] = dataset.apply(lambda row: sent2features(row['tokens'], row['pos_list']), axis=1)

In [9]:
dataset["features"].sample(1, random_state=12).iloc[0]

[{'bias': 1.0,
  'word.lower()': 'had',
  'word[-3:]': 'Had',
  'word[-2:]': 'ad',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'NNP',
  'postag[:2]': 'NN',
  'BOS': True,
  '+1:word.lower()': 'my',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'PRP$',
  '+1:postag[:2]': 'PR'},
 {'bias': 1.0,
  'word.lower()': 'my',
  'word[-3:]': 'my',
  'word[-2:]': 'my',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'postag': 'PRP$',
  'postag[:2]': 'PR',
  '-1:word.lower()': 'had',
  '-1:word.istitle()': True,
  '-1:word.isupper()': False,
  '-1:postag': 'NNP',
  '-1:postag[:2]': 'NN',
  '+1:word.lower()': 'hubby',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'NN',
  '+1:postag[:2]': 'NN'},
 {'bias': 1.0,
  'word.lower()': 'hubby',
  'word[-3:]': 'bby',
  'word[-2:]': 'by',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'p

In [10]:
nlp = spacy.load("en_core_web_sm")

equivalence_table = {"PERSON":"-per",
                     "NORP":"-grp",
                     "FAC":"-fac",
                     "ORG":"-org",
                     "GPE":"-gpe",
                     "LOC":"-geo",
                     "DATE":"-date",
                     "TIME":"-tim",
                     "WORK_OF_ART":"-art",
                     "LAW":"-law",
                     "LANGUAGE":"-lang",
                     "EVENT":"eve",
                     "PRODUCT":"-prod",
                     "PERCENT":"-pct",
                     "MONEY":"-mon",
                     "QUANTITY":"-qty",
                     "CARDINAL":"-card",
                     "ORDINAL":"-ord",
                     "":""
                     }

In [11]:
dataset["ner_labels_custom"] = dataset.apply(
    lambda row: align_bio_to_custom_tokens(
        row["text"], 
        row["tokens"],
        nlp,
        equivalence_table
    ),
    axis=1
)

In [12]:
# Sanity check
dataset[['text','tokens','ner_labels_custom']].head()

,text,tokens,ner_labels_custom
0,"One word amazing!! The red fish, halibut, fr...","[One, word, amazing, !, !, The, red, fish, ,, ...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,First time here and the food is great and the ...,"[First, time, here, and, the, food, is, great,...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,I recently had the pleasure of dining at Optim...,"[I, recently, had, the, pleasure, of, dining, ...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,Beautiful atmosphere and delicious food. All o...,"[Beautiful, atmosphere, and, delicious, food, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,We had a wonderful dinner at the Optimist. Our...,"[We, had, a, wonderful, dinner, at, the, Optim...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."


In [ ]:
# # Expand rows so each token becomes its own row
# dataset["ner_labels"] = dataset.apply(
#     lambda row: align_bio(nlp(row["text"]), row["tokens"]),
#     axis=1
# )

In [16]:
# dataset.head()

## <font color='#BFD72F' size=6>3.2 Model Implementation</font> <a class="anchor" id="3_2"></a>
  
[Back to TOC](#toc)

In [17]:
# Quick (q1) train-test split that we can use to test our classifier
X_train_q1, X_test_q1, y_train_q1, y_test_q1 = train_test_split(
                                                        dataset["features"], 
                                                        dataset["ner_labels_custom"],
                                                        test_size=0.2,
                                                        random_state=39
                                                        )

train_index_q1 = list(X_train_q1.index)
test_index_q1 = list(X_test_q1.index)

In [18]:
# CRF model -> conditional random field model

crf = sklearn_crfsuite.CRF(algorithm='lbfgs', 
                           c1=0.1, 
                           c2=0.1, 
                           max_iterations=100, 
                           all_possible_transitions=True,
                           verbose=True 
                           )
try:
    crf.fit(X_train_q1, y_train_q1)
except AttributeError:
    pass

loading training data to CRFsuite: 100%|██████████| 42852/42852 [00:10<00:00, 3908.38it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 137043
Seconds required: 2.105

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=7.01  loss=3408241.81 active=136264 feature_norm=1.00
Iter 2   time=7.20  loss=2161165.47 active=136021 feature_norm=14.90
Iter 3   time=3.60  loss=2015023.94 active=124973 feature_norm=14.04
Iter 4   time=62.01 loss=462172.08 active=84138 feature_norm=5.89
Iter 5   time=29.20 loss=385031.43 active=97761 feature_norm=5.88
Iter 6   time=61.40 loss=382929.00 active=101881 feature_norm=5.68
Iter 7   time=89.85 loss=372978.09 active=107747 feature_norm=5.67
Iter 8   time=100.48 loss=371056.01 active=108125 feature_norm=5.71
Iter 9   time=91.06 loss=368006.23 active=104889 feature_n

In [19]:
# keep tokens for inspection
X_test_tokens_q1 = dataset["tokens"].loc[test_index_q1]
y_pred_q1 = crf.predict(X_test_q1)

In [ ]:
# # save model
# import joblib

# models_dir = os.path.join('..','models')
# os.makedirs(models_dir, exist_ok=True)
# joblib.dump(crf, os.path.join(models_dir, 'crf_ner_template.pkl'))

## <font color='#BFD72F' size=6>3.3 Model Evaluation</font> <a class="anchor" id="3_3"></a>
  
[Back to TOC](#toc)

In [20]:
labels = list(crf.classes_)
labels.remove("O")

round(metrics.flat_f1_score(y_test_q1, y_pred_q1,average='weighted', labels=labels), 3)

0.725

In [ ]:
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)

print(sklearn_crfsuite.metrics.flat_classification_report(y_test_q1, y_pred_q1, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

       B-art      0.548     0.193     0.286        88
       I-art      0.420     0.210     0.280       100
      B-card      0.824     0.863     0.843      2122
      I-card      0.833     0.723     0.774       256
      B-date      0.845     0.832     0.839      1389
      I-date      0.846     0.833     0.839      1224
       B-fac      0.535     0.349     0.422       109
       I-fac      0.532     0.365     0.433       137
       B-geo      0.549     0.411     0.470        95
       I-geo      0.273     0.154     0.197        39
       B-gpe      0.787     0.707     0.745       998
       I-gpe      0.799     0.728     0.762       180
       B-grp      0.813     0.789     0.801       968
       I-grp      0.750     0.514     0.610        35
      B-lang      0.500     0.692     0.581        13
       B-law      0.462     0.300     0.364        20
       I-law      0.229     0.250     0.239        32
       B-mon      0.827    